# Example: dispatch-derived LCOE and TEA

This standalone example loads the full 2023 PNM profiles, declares PV and battery-storage economics, dispatches the annual system, and passes operation metrics to `LCOECalculator`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'enliten').is_dir():
    if root.parent == root: raise RuntimeError('Run from inside the ENLITEN repository.')
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from enliten import ChargingPath, Generation, LCOECalculator, Site, Storage, System

def profile(filename):
    frame = pd.read_csv(root / 'examples' / 'data' / filename)
    return pd.Series(frame['PNM'].to_numpy(float), index=pd.to_datetime(frame.iloc[:, 0], utc=True))

demand, pv = profile('PNM_demand.csv'), profile('PNM_pv_ac_1MW_av.csv')
assert demand.index.equals(pv.index)
load = (demand * 0.05).rename('load_MW')
pv_multiplier, bes_capacity, bes_power = 3_000.0, 750.0, 150.0
pv_capex = pv.max() * pv_multiplier * 1_000 * 1_430
pv_opex = pv.max() * pv_multiplier * 1_000 * 24
bes_capex = bes_capacity * 1_000 * 300
site = Site('microgrid')
pv_system = Generation('pv', site, pv * pv_multiplier, 'electric', capex=pv_capex, opex=pv_opex)
bes = Storage('bes', site, bes_capacity, bes_power, 'electric', 'electric', 0.90, maximum_stored_energy_rate_MW=bes_power, capex=bes_capex, opex=0.025 * bes_capex)
path = ChargingPath('pv', 'bes', 'electric', 'electric', 0.90, bes_power / 0.90)
system = System(load, [bes, pv_system], [path])
system.operation_metrics()

{'operating_hours': 8759,
 'load_MWh_electric': 714158.81,
 'generation_to_load_MWh_electric': 305903.8760946745,
 'storage_to_load_MWh_electric': 118657.7619851093,
 'system_to_load_MWh_electric': 424561.6380797839,
 'grid_to_load_MWh_electric': 289597.1719202162,
 'unmet_load_MWh_electric': 0.0,
 'export_energy_MWh_electric': 0.0,
 'percent_load_met': 99.99999999999999,
 'percent_load_by_system': 59.449191431214565,
 'system_capex_USD': 440580965.5543319,
 'system_annual_OM_USD': 9243142.079233542,
 'pv_curtailed_MWh_electric': 0.0,
 'load_annual_MWh_electric': [714240.3442858774],
 'system_to_load_annual_MWh_electric': [424610.1095534772],
 'grid_to_load_annual_MWh_electric': [289630.2347324003],
 'export_energy_annual_MWh_electric': [0.0],
 'annual_electricity_sales_USD': [0.0],
 'annual_electricity_purchases_USD': [0.0],
 'system_annual_VOM_USD': 0.0,
 'system_augment_USD': [0.0],
 'system_augment': [0.0]}

In [2]:
pd.DataFrame([{'asset': asset.name, 'capex_USD': asset.capex, 'fixed_OM_USD_per_year': asset.opex, 'variable_OM_USD_per_MWh': asset.variable_opex_USD_per_MWh} for asset in system.systems])

,asset,capex_USD,fixed_OM_USD_per_year,variable_OM_USD_per_MWh
0,bes,2.250000e+08,5.625000e+06,0.0
1,pv,2.155810e+08,3.618142e+06,0.0


In [3]:
lcoe_metrics = LCOECalculator.from_system(system, analysis_period=30).calculate_lcoe_metrics()
pd.Series(lcoe_metrics)

PVD                     7.673733e-01
FCR_AT                  3.614807e-02
FCR_BT                  5.294061e-02
CRF                     7.650023e-02
NPV_cash_flow          -4.598092e+08
IRR                              NaN
payback_period                   NaN
LCOE_real_USD_kWh_BT    8.521855e-02
LCOE_real_USD_kWh_AT    6.001072e-02
dtype: float64